In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import torch.optim as optim
import gymnasium as gym
import numpy as np
from torch.distributions import Categorical

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.layer1 = nn.Linear(obs_dim, 128)
        self.layer2 = nn.Linear(128, 128)
        self.actor = nn.Linear(128, act_dim)
        self.critic = nn.Linear(128, 1)

    def forward(self, state):
        x = self.layer1(state)
        x = F.relu(x)
        x = self.layer2(x)
        x = F.relu(x)
        val = self.critic(x)
        logits = self.actor(x)

        return val, logits


        



In [ ]:
class Buffer:
    def __init__(self, rollout_len, obs_dim):
        self.rollout_len = rollout_len
        self.curr = 0

        self.observations = torch.zeros((rollout_len, obs_dim))
        self.actions = torch.zeros(rollout_len, dtype=torch.long)
        self.values = torch.zeros(rollout_len)
        self.rewards = torch.zeros(rollout_len)
        self.dones = torch.zeros(rollout_len)

    def push(self, observation, action, val, reward, done):
        self.observations[self.curr] = observation
        self.actions[self.curr] = action
        self.values[self.curr] = val
        self.rewards[self.curr] = reward
        self.dones[self.curr] = done
        self.curr += 1
    
    def is_full(self):
        if self.curr >= self.rollout_len:
            return True
        return False

    def reset(self):
        self.curr = 0

    


_IncompleteInputError: incomplete input (553541573.py, line 26)

In [ ]:
class Agent:
    def __init__(self, env):
        self.network = ActorCritic(env.observation_space.shape[0], env.action_space.n)
        self.discount = 0.99
        self.T = 32
        self.c_v = 0.5
        self.entropy = 0.01
        self.lr = 0.0007
        self.gradient_clip = 0.5
        self.buffer = Buffer(self.T, env.observation_space.shape[0])
        self.env = env
        self.optimizer = torch.optim.AdamW(self.network.parameters(), lr=self.lr, weight_decay=0)

    def collect_rollout(self, observation):
        observation = torch.as_tensor(observation, dtype=torch.float32)
        with torch.no_grad():
            val, logits = self.network.forward(observation)
        dist = Categorical(logits=logits)
        action = dist.sample().item()

        next_observation, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        self.buffer.push(observation, action, val.item(), reward, done)

        return next_observation, done

    def compute_targets(self, bootstrap):
        R = bootstrap
        return_targets = torch.zeros(self.T)
        for i in range(self.T - 1, -1, -1):
            R = self.buffer.rewards[i] + self.discount * R * (1.0-self.buffer.dones[i])
            return_targets[i] = R
        advantages = return_targets - self.buffer.values

        return advantages, return_targets

    def update(self, advantages, targets):
        values, logits = self.network(self.buffer.observations)
        values = values.squeeze(-1)
        dist = Categorical(logits=logits)
        log_probs = dist.log_prob(self.buffer.actions)

        policy_loss = -(log_probs * advantages.detach()).mean()
        value_loss = F.mse_loss(values, targets)
        entropy = dist.entropy().mean()
        loss = policy_loss + self.c_v * value_loss - self.entropy * entropy

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.network.parameters(), self.gradient_clip)
        self.optimizer.step()
        self.buffer.reset()

_IncompleteInputError: incomplete input (3627362429.py, line 29)